# 👑🐱 喵皇御書房｜打造貓咪飼養知識向量資料庫


把整理好的貓咪飼養知識文字塊（38 份 `.md`）轉成向量，建立 FAISS 向量資料庫，最後打包成 **`rag-data.zip`** 給對話機器人使用。

| 項目 | 老師範例 | 我的版本 |
|---|---|---|
| 資料 | 校園法規 PDF，自動每 500 字切 | 農委會《貓隻飼養與照顧指南》等 → **改寫成 38 個主題文字塊** |
| 切塊方式 | `RecursiveCharacterTextSplitter` | **一個主題 = 一個文字塊**，splitter 只當超長保險 |
| Metadata | 只有檔名 | **標題、分類、來源、網址**（講義 p.14～16） |
| Embedding 前綴 | `title: none` | **`title: {真實標題}`**（ |
| 讀取資料 | 手動上傳 | **自動從雲端網址下載** |

## 0. 下載我的原始資料 `cat_docs.zip`

> **注意**：下面網址就是我的原始資料（Google Drive 公開分享連結），直接執行即可，不用手動上傳。

In [ ]:
!pip -q install gdown

In [ ]:
# 我的原始資料（38 份貓咪飼養知識文字塊）的雲端分享連結
DOCS_ZIP_URL = "https://drive.google.com/file/d/1y10LxYBEBjfpkkpOKI3XHDTmPKtdRKxw/view?usp=sharing"

In [ ]:
import os
if DOCS_ZIP_URL.startswith("http"):
    !gdown --fuzzy -O cat_docs.zip "{DOCS_ZIP_URL}"
    !unzip -o -q cat_docs.zip
else:
    print("⚠️ 還沒填網址：請手動把 cat_docs.zip 上傳到左側檔案區再繼續")
    if os.path.exists("cat_docs.zip"):
        !unzip -o -q cat_docs.zip

upload_dir = "cat_docs"
print("文字塊數量：", len([f for f in os.listdir(upload_dir) if f.endswith(".md")]))

Downloading...
From: https://drive.google.com/uc?id=1y10LxYBEBjfpkkpOKI3XHDTmPKtdRKxw
To: /content/cat_docs.zip
100% 33.7k/33.7k [00:00<00:00, 44.2MB/s]
文字塊數量： 38


## 1. 更新必要套件並引入

In [ ]:
!pip install -U langchain langchain-community pypdf python-docx faiss-cpu
!pip install -U sentence-transformers transformers
!pip install -U langchain-huggingface langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.5 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 16.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 53.5 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transform

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import re, json

/tmp/ipykernel_1571/431283289.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredWordDocumentLoader


## 2. 依 Google 建議加入 EmbeddingGemma 前綴

Google 對 EmbeddingGemma 的建議格式：

    文件： title: {title | none} | text: ...
    問題： task: search result | query: ...

In [ ]:
class EmbeddingGemmaEmbeddings(HuggingFaceEmbeddings):
    def __init__(self, **kwargs):
        super().__init__(
            model_name="google/embeddinggemma-300m",       # HF 上的官方模型
            encode_kwargs={"normalize_embeddings": True},  # 正規化 → 內積就是 cosine
            **kwargs
        )

    def embed_documents(self, texts):
        texts = [f"title: none | text: {t}" for t in texts]
        return super().embed_documents(texts)

    # 帶入每個文字塊自己的標題
    def embed_documents_with_titles(self, texts, titles):
        texts = [f"title: {ti} | text: {t}" for t, ti in zip(texts, titles)]
        return HuggingFaceEmbeddings.embed_documents(self, texts)

    def embed_query(self, text):
        return super().embed_query(f"task: search result | query: {text}")

## 3. 載入文件並解析 metadata

每個 `.md` 檔都是我依照講義的格式整理的自訂文字塊：

```
標題：貓不能吃的人類食物          ← metadata
分類：飲食                        ← metadata（RAG02 可用來篩選）
來源：行政院農業委員會《貓隻飼養與照顧指南》
來源網址：https://...
【內容】  ...                      ← 真正拿去 embedding 的部分
【檢索練習問題】 貓可以吃葡萄嗎？ ...  ← 也一起 embedding，讓「問句」更容易對上
【提醒】 ...
```

> 在文字塊裡先放幾個可能的問題，問句和問句的向量會更接近，檢索命中率更高。

In [ ]:
def parse_chunk(raw: str, filename: str):
    """把一個 .md 文字塊拆成：要 embedding 的文字 + metadata"""
    meta = {"filename": filename}
    for key, field in [("標題", "title"), ("分類", "category"),
                       ("來源", "source"), ("來源網址", "url")]:
        m = re.search(rf"^{key}：(.+)$", raw, re.M)
        meta[field] = m.group(1).strip() if m else ""

    def section(name):
        m = re.search(rf"【{name}】\n(.*?)(?=\n【|\Z)", raw, re.S)
        return m.group(1).strip() if m else ""

    content = section("內容")
    questions = section("檢索練習問題")
    meta["questions"] = questions.replace("\n", "／")
    page_content = f"{content}\n\n（常見問題：{meta['questions']}）"
    return page_content, meta

In [ ]:
documents = []
for file in sorted(os.listdir(upload_dir)):
    path = os.path.join(upload_dir, file)
    if file.endswith(".md"):                      # 我的自訂文字塊
        with open(path, encoding="utf-8") as f:
            text, meta = parse_chunk(f.read(), file)
        documents.append(Document(page_content=text, metadata=meta))
    # 保留老師的寫法：之後要再丟 txt / pdf / docx 進來也可以
    elif file.endswith(".txt"):
        documents.extend(TextLoader(path, encoding="utf-8").load())
    elif file.endswith(".pdf"):
        documents.extend(PyPDFLoader(path).load())
    elif file.endswith(".docx"):
        documents.extend(UnstructuredWordDocumentLoader(path).load())

print(f"共載入 {len(documents)} 份文件")
print(documents[21].metadata)
print(documents[21].page_content[:200])

共載入 38 份文件
{'filename': '22_貓不能吃的人類食物.md', 'title': '貓不能吃的人類食物', 'category': '飲食', 'source': '行政院農業委員會《貓隻飼養與照顧指南》', 'url': 'https://www.es.nccu.edu.tw/wp-content/uploads/sites/30/2023/03/貓隻飼養與照顧指南.pdf', 'questions': '貓可以吃葡萄嗎？／貓可以吃巧克力嗎？／可以給貓吃我的便當嗎？／中秋節可以幫貓戴柚子帽嗎？／貓可以吃蔥嗎？'}
不宜餵貓的人類食物：
・蔬菜：洋蔥、青蔥、大蒜。
・水果：櫻桃、葡萄、楊桃、桃子、酪梨、柚子、柑橘類。
・其他：夏威夷豆等堅果、含木醣醇的口香糖、巧克力；過鹹、過油或含咖啡因的食物與辛香料。
・中秋節的「柚子帽」也可能危害健康。
・骨頭、便當剩菜中的堅硬殘渣可能刺傷口腔和消化道；受汙染或腐敗的食物也會致病。
如果貓咪誤食，請盡快聯絡動物醫院。

（常見問題：貓可以吃葡萄嗎？／貓可以吃巧克力嗎？／可


In [ ]:
# 看一下各分類有幾個文字塊
from collections import Counter
Counter(d.metadata.get("category", "其他") for d in documents)

Counter({'認識貓咪': 6,
         '飼養準備': 2,
         '法規': 3,
         '居家環境': 5,
         '安全防護': 4,
         '飲食': 4,
         '健康保健': 4,
         '行為社交': 4,
         '幼貓照顧': 2,
         '高齡貓照顧': 2,
         '多貓家庭': 2})

## 4. 切塊當保險

我的文字塊已經是一個主題一塊，長度都在 500 字內，所以 splitter 幾乎不會再切；這裡只是保險，如果之後加入很長的 txt / pdf，還是會被切成 500 字、重疊 100 字。

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100,
                                          separators=["\n\n", "\n", "。", "，", ""])
split_docs = splitter.split_documents(documents)
print("切塊前：", len(documents), "→ 切塊後：", len(split_docs))
print("最長文字塊字數：", max(len(d.page_content) for d in split_docs))

切塊前： 38 → 切塊後： 38
最長文字塊字數： 326


## 5. 建立向量資料庫

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HuggingFace')

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
embedding_model = EmbeddingGemmaEmbeddings()

texts  = [d.page_content for d in split_docs]
titles = [d.metadata.get("title") or d.metadata.get("filename", "none") for d in split_docs]
metas  = [d.metadata for d in split_docs]

vectors = embedding_model.embed_documents_with_titles(texts, titles)
vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, vectors)),
    embedding=embedding_model,
    metadatas=metas,
)
print("向量維度：", len(vectors[0]), "｜向量數：", vectorstore.index.ntotal)

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

... (11 outputs left out)

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

向量維度： 768 ｜向量數： 38


### 5.1 小測試：檢索看看準不準

建完先自己考一下資料庫，確認問題 → 文字塊對得上。
分數是 cosine 相似度（越接近 1 越像）。

In [ ]:
def quick_test(q, k=3):
    print(f"\n❓ {q}")
    for doc, dist in vectorstore.similarity_search_with_score(q, k=k):
        cos = 1 - dist / 2            # 向量已正規化：L2² 距離 → cosine
        print(f"   {cos:.3f}  [{doc.metadata['category']}] {doc.metadata['title']}")

for q in ["貓可以吃葡萄嗎", "我家公貓一直蹲砂盆尿不出來",
          "沒幫貓植入晶片會被罰嗎", "兩隻貓要準備幾個貓砂盆"]:
    quick_test(q)


❓ 貓可以吃葡萄嗎
   0.667  [飲食] 貓不能吃的人類食物
   0.520  [認識貓咪] 貓是純肉食動物：不能吃全素
   0.487  [飲食] 飲食基本原則與換飼料

❓ 我家公貓一直蹲砂盆尿不出來
   0.740  [健康保健] 急症：公貓尿不出來（尿道阻塞）
   0.636  [居家環境] 貓砂盆的數量、位置與清潔
   0.560  [飲食] 飲水：讓貓多喝水的方法

❓ 沒幫貓植入晶片會被罰嗎
   0.603  [法規] 2025 新制：貓咪也要強制寵物登記
   0.540  [法規] 棄養與虐待的罰則（含 2026 修法草案）
   0.503  [飼養準備] 飼主的三大基本任務：疫苗、絕育、晶片登記

❓ 兩隻貓要準備幾個貓砂盆
   0.676  [居家環境] 貓砂盆的數量、位置與清潔
   0.601  [多貓家庭] 新舊貓磨合與多貓共養
   0.579  [多貓家庭] 第二隻貓進家門前的準備


## 6. 儲存並打包成 `rag-data.zip`

`faiss_db/` 裡會有 `index.faiss`（向量）和 `index.pkl`（文字＋metadata）。
另外存一份 `chunks.json` 方便人工檢查。

In [ ]:
vectorstore.save_local("faiss_db")

with open("faiss_db/chunks.json", "w", encoding="utf-8") as f:
    json.dump([{"text": t, **m} for t, m in zip(texts, metas)], f, ensure_ascii=False, indent=1)

!rm -f rag-data.zip
!zip -r rag-data.zip faiss_db

  adding: faiss_db/ (stored 0%)
  adding: faiss_db/index.faiss (deflated 7%)
  adding: faiss_db/chunks.json (deflated 69%)
  adding: faiss_db/index.pkl (deflated 64%)


In [ ]:
from google.colab import files
files.download("rag-data.zip")
print("✅ 向量資料庫已打包為 'rag-data.zip'。請上傳到 Google Drive、設定『知道連結的人皆可檢視』，把連結貼到 RAG02。")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 向量資料庫已打包為 'rag-data.zip'。請上傳到 Google Drive、設定『知道連結的人皆可檢視』，把連結貼到 RAG02。
